# NB09 — Zero-shot Sensitivity Demonstration (base SegEarth-OV-3 knobs)

**Runs on Kaggle T4 GPU.**

This is **not** the NB06/NB08 4-method ablation (ZS-single / ZS-multi / +PAMR /
PTSAM+PAMR) — that compares *methods*. This notebook proves that the **base
zero-shot approach's own knobs** (window size, `prob_thd`/`confidence_threshold`,
and prompt wording) each independently move the segmentation result. Each
config variant is rendered as its own full-tile RGB | overlay image — same
format as the teammate's reference outputs (`img_size`/`prob_thd`/`conf_thd`/
`slide_stride`/`slide_crop` footer, own class-color legend row, α=0.6 overlay)
— not a subplot comparison grid, so every result is inspectable at full
resolution on its own.

- **Image reference A** — window-size sweep: `(slide_crop, slide_stride)` ∈
  `{(768,576), (1024,768), (1500,1200)}`, prompts/thresholds held at baseline.
- **Image reference B** — `prob_thd` sweep (free, reuses baseline logits) and
  `confidence_threshold` sweep (real reruns), window size held at baseline.
- **Image reference C** — prompt-wording sweep: single-word vs multi-synonym
  (baseline) vs deliberately ambiguous, to visibly show class confusion (e.g.
  runway collapsing into road).

**3 images, deliberately picked to stress SAM3 outside the dense-building
regime NB03/NB08 already cover well:**

| # | Tile | Dataset | Scene | Headlines |
|---|---|---|---|---|
| 1 | `dop20_32_468_5543_1_he` | `dummyirl/frankfurt-dot20` | Airport / runway / apron | Part C (runway vs road prompt ambiguity), Part B (uniform tarmac → threshold-sensitive) |
| 2 | `dop20_32_475_5550_1_he` | `dummyirl/frankfurt-dot20` | Frankfurt Hbf — rail tracks, station, platforms | Part B (thin track bundles → crop/stride-sensitive), Part C (rail vs road wording) |
| 3 | `dop20_32_476_5524_1_he` | `dummyirl/darmstadt-dop20` | Water, buildings, sports pitch/court, roads (same tile as NB03/NB08) | Part B (pitch/stand boundary), Part C (grass vs pitch wording) |

**Datasets to attach:** `dummyirl/sam3-weights`, `dummyirl/frankfurt-dot20`,
`dummyirl/darmstadt-dop20`.

> **Slug note:** the Frankfurt dataset's actual Kaggle slug is
> `frankfurt-dot20` (typo in the teammate's dataset name — "dot" not "dop"),
> confirmed via `kaggle datasets list --user dummyirl`. An earlier push of
> this notebook used the plausible-looking but wrong `frankfurt-dop20` and
> Kaggle silently dropped that dataset from `dataset_sources` (with a CLI
> warning) — the tile-resolution code below still works either way since it
> globs by filename, not by dataset slug, but the actual image files need the
> dataset attached to be findable at all.

**Compute strategy:** `prob_thd` is a post-hoc threshold on already-computed
logits (`segearthov3_segmentor.py:215`) — swept for free from one cached
logit map per image. `confidence_threshold` and window size change what
SAM3 actually computes, so each value is a real rerun. Prompt-set sweeps are
also real reruns; the multi-synonym panel reuses the baseline run. Images are
processed **sequentially** — a single T4 has one GPU, so "parallel" images
would just interleave the same GPU work for no benefit.

> Visual comparison only — no ground truth for these tiles, so no mIoU/F1.

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX \u2192 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated \u2192 {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned \u2192 {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Part B + Part C: sensitivity sweeps on 3 full-tile images

Backbone runs once per crop per config variant. `prob_thd` variants reuse
the baseline logits (free); `confidence_threshold`, `slide_stride`, and
prompt-set variants each require their own sliding-window pass.

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
# Set this to one stem to run only that tile (kept separate pushes fast).
# Empty string = all tiles (slow, not recommended).
export TARGET_TILE="dop20_32_476_5524_1_he"
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, os, json, torch, torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE   = "cuda"
OUT_DIR  = Path("/kaggle/working/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = OUT_DIR / "preds"; PRED_DIR.mkdir(parents=True, exist_ok=True)
TARGET_TILE = os.environ.get("TARGET_TILE", "").strip()

# ── default baseline params (NB03's known-good config for 5000x5000 tiles) ──
# Per-tile "baseline" overrides this via IMAGE_CONFIGS[stem]["baseline"] when
# present (tile 4 uses the teammate's exact published params for direct
# comparability — see that tile's comment below).
DEFAULT_BASELINE = dict(prob_thd=0.1, confidence_threshold=0.1, slide_crop=1024, slide_stride=768)
PROB_THD_SWEEP     = [0.05, 0.1, 0.3]
CONF_THD_SWEEP     = [0.05, 0.1, 0.3]
# Default window-size sweep; per-tile override via cfg["window_sweep"] (see
# tile 3's config below for the one tile currently using this).
WINDOW_SIZE_SWEEP  = [(768, 576), (1024, 768), (1500, 1200)]  # (slide_crop, slide_stride)
# NOTE: a full-tile crop=256/stride=128 pass was tried (v3/v4) and pulled —
# ~1521 crops/tile x 8 sequential per-class grounding calls each ran well
# past 3h with no completion signal from Kaggle's non-streaming log capture.
# Small-window testing needs its own notebook on a cropped sub-region, not
# a full 5000x5000 tile, before ever putting it in this sweep again.
# Reference: teammate's own window-size sweep (D:\Downloads\drive-download-
# 20260718T135518Z-1-001\dop20_32_472_5525_1_he\windowsize_exp_extracted\)
# used crop/stride pairs from 400/350 up to 1500/1500 — 400/350 was visibly
# noisier, 1500/1500 left black dropout holes in the forest class. This
# sweep's 3 points (768/576, 1024/768, 1500/1200) sample that same range.
BG_IDX = 255

# ── per-image config: multi-synonym (baseline) + single-word + ambiguous prompts ──
# "multi" is what's sent to SAM3 for grounding (long, descriptive — helps
# detection). "display" is what's shown in the legend/rendering (short,
# 1-2 words — per teammate's reference style, kept out of the plotting code
# entirely so a full multi-synonym string never leaks into a saved image).
IMAGE_CONFIGS = {
    # Reworked after v1: v1's tarmac synonym list still lost coverage at
    # prob_thd=0.3 because none of its phrases described the runway/apron's
    # actual texture (faded painted markings, patched/worn asphalt) — SAM3's
    # text side had nothing to anchor low-confidence tarmac pixels to, so
    # they fell below threshold and dropped to background. Added markings/
    # patched-surface wording to give more low-level visual detail to match.
    # Reworked again after v6: at confidence_threshold=0.3 and with
    # single-word prompts, the runway/tarmac area rendered as a flat
    # white/light-grey blob (SAM3 grounded on the wrong region, not just
    # low-confidence dropout) — added even more concrete texture/color
    # cues (concrete slab joints, painted centerline, oil stains) to give
    # SAM3's text side something to lock onto beyond generic "grey surface".
    # Reworked again after v6 baseline comparison against the teammate's
    # reference on this exact tile: even at baseline params the runway/
    # apron/taxiway area rendered as fully transparent background (not
    # just weak) — the class was losing to background everywhere, not
    # losing to another class. Also dropped "box, container, cargo" and
    # "tree, forest" from this tile's class list: neither ever appears in
    # this scene, so they were dead legend entries with zero true-positive
    # pixels, just adding legend clutter and consuming one of 8 grounding
    # passes for nothing.
    # Reworked again after v7: strengthening the runway prompt further
    # (v6->v7) had zero effect on the output image — checked the actual
    # per-class pixel histogram of the saved prediction and runway had
    # exactly 0 px, with road alone claiming 21.8% of the tile. This is
    # class confusion (road winning the argmax everywhere), not a
    # coverage/threshold problem — no amount of runway wording can fix
    # that if the competing class is the one being too permissive.
    # Narrowed the road prompt to road-specific cues (painted lane
    # markings, curbs, passing traffic) that a runway/apron doesn't have,
    # instead of generic "dark grey asphalt road" which sat too close to
    # "light grey paved airfield surface" in SAM3's text embedding space.
    "dop20_32_468_5543_1_he": {
        "multi": ["airfield runway, taxiway, apron, light grey paved airfield surface, tarmac with faded painted markings, concrete slab joints, painted centerline stripes, oil stains",
                   "public paved road with painted lane markings and curbs, street with passing cars",
                   "airplane, aircraft, white plane",
                   "car, vehicle, truck",
                   "building, rooftop, solar panel roof, vent, roof equipment",
                   "low vegetation, grass, field"],
        "display": ["runway", "road", "aircraft", "vehicles", "building", "grass"],
        "single": ["runway", "road", "aircraft", "car", "building", "grass"],
        "ambiguous": ["paved surface", "paved surface", "vehicle", "vehicle",
                      "structure", "grass"],
        "colors": np.array([
            [220, 220, 220],   # runway/taxiway — light grey
            [ 60,  60,  60],   # road — dark grey
            [255, 165,   0],   # aircraft — orange
            [255, 255,   0],   # car — yellow
            [  0,   0, 255],   # building — blue
            [  0, 255, 255],   # grass/low veg — cyan
        ], dtype=np.uint8),
    },
    # Reworked after v1: v1's "platform strip" wording was too close to
    # "paved road, street" for SAM3's text encoder, so platforms sometimes
    # got read as road. Added concrete visual cues (raised edge, yellow
    # tactile strip) that don't apply to roads, to separate the two classes.
    # Reworked again after tile-2 run (v2): with the narrowed road prompt in
    # place, railway/platform/road were ALL still at 0 px — checked the
    # pixel histogram and grass alone claimed 11.7% while background sat
    # at 59.7%, and visually grass (cyan) had bled onto clearly-urban/paved
    # areas that aren't grass. Same failure class as tile 1's road-swallows-
    # runway problem: "low vegetation, grass" is too generic and its text
    # embedding apparently matches enough of the scene's grey/mixed-texture
    # areas to dominate the argmax. Narrowed to color/texture-specific
    # wording that shouldn't match paved or ballast surfaces.
    # Reworked again after inspecting v2's Part-C single-word variant:
    # the bare "railway" single-word prompt found 10.9% railway coverage
    # (patchy but real) where the elaborate multi-synonym prompt found 0% —
    # opposite of the runway lesson from tile 1, confirming there's no
    # universal "always longer" or "always shorter" rule, it depends on the
    # specific class. Switched railway's multi prompt to short, concrete
    # wording.
    # Reworked again after the user described the actual source image:
    # this tile is shot in strong sun with overexposure/shadow — road,
    # rail track ballast, and platform surfaces likely sit in a visually
    # similar washed-out light-grey tonal range in THIS specific image, not
    # just similar-sounding words. Also confirmed: no water in this tile,
    # vehicles present but small/sparse.
    # Reworked again per user call: dropped platform as a class for this
    # tile (not confident it's necessary/worth a class slot here — can
    # revisit later if railway/road land cleanly first). Reverted tree
    # wording back to plain "tree" (dense-canopy framing not needed for
    # this scene's scattered trees). Rewrote road wording to not assume
    # normal dark-asphalt contrast — the overexposure affects road same as
    # platform/ballast, so lane-marking/curb cues alone may not be visible
    # either; added washed-out/bright-grey framing so the prompt matches
    # what the road plausibly looks like in this specific lighting instead
    # of a generic road description.
    # v4 result: railway short-phrase fix worked (0% -> 11.3%), but only
    # over open trackbed — the shed-covered platform section stayed
    # background. Road barely moved (0% -> 0.5%) despite single-word road
    # alone finding 32.9% in the same run. User then examined the source
    # image directly and reported ~12 platform "stripes" fanning out of
    # the Hbf building, with trains actually parked in the tracks, and
    # made the call: (1) stop treating railway/platform as separate
    # concepts — segment the WHOLE trackbed (rails + ballast + platform
    # islands) as one merged class, since that whole area is one
    # continuous visual surface in this image, not two; (2) add a
    # dedicated `train` class for the rolling stock sitting on the
    # tracks, which had no prompt at all before and was silently falling
    # into railway/building/background; (3) user also independently
    # spotted that C_prompt_single-word visibly recovered small
    # road-colored buildings that every other variant (including our
    # "proper" multi-synonym baseline) mislabeled as road — a 4th
    # confirmed instance of short wording beating descriptive wording on
    # THIS specific tile (previously seen on railway, road coverage
    # amount, and grass/tree completeness). Given that consistent
    # pattern, every class prompt below is now deliberately kept short/
    # plain rather than descriptive, instead of reserving that only for
    # railway as before.
    # v5 result: `train` worked (0% -> 1.0%, visually confirmed as real
    # rolling-stock detection). But the merged `tracks` class (railway+
    # platform combined into one prompt) regressed to 0% — worse than v4's
    # `railway`-alone result of 11.3%. Road swung to 42.5% (from 0.5%),
    # plausibly overcorrected. User reviewed C_prompt_ambiguous directly
    # and made the final call: DROP tracks/railway as a class entirely —
    # it isn't reliably detectable on this tile as its own concept no
    # matter the wording tried (short, long, merged). The open ballast/
    # trackbed area is left unclassified (background) rather than forced
    # into any class; only `train` (sitting on top of it) and `building`
    # (the platform/shed structure — user confirmed the ambiguous variant's
    # pattern of building extending into the platform/canopy area is
    # correct and desired, since they're physically attached) get labels
    # there. User also flagged a separate problem visible in the ambiguous
    # variant: low vegetation/grass is being absorbed into `trees` at some
    # points (grass basically disappeared, 0.1%, while trees picked up
    # some of that area) — reworded `trees` back toward woody/canopy
    # framing to disambiguate it from lower grass/lawn vegetation, since
    # the two were made too similar by both being reduced to single generic
    # words in the v5 short-prompt-everywhere pass.
    "dop20_32_475_5550_1_he": {
        "multi": ["building",
                   "road",
                   "car",
                   "tree, wooded canopy",
                   "grass, lawn, low vegetation",
                   "train"],
        "display": ["building", "road", "vehicles", "trees", "grass", "train"],
        "single": ["building", "road", "car", "tree", "grass", "train"],
        "ambiguous": ["structure", "paved surface",
                      "vehicle", "grass", "grass", "vehicle"],
        "colors": np.array([
            [  0,   0, 255],   # building — blue
            [ 60,  60,  60],   # road — dark grey
            [255, 255,   0],   # car — yellow
            [  0, 200,   0],   # tree — green
            [  0, 255, 255],   # grass — cyan
            [255, 140,   0],   # train — dark orange
        ], dtype=np.uint8),
    },
    # Reworked after v1: v1's "clay sports court, dirt sports ground, tennis
    # court" and "green sports field, football pitch, grass pitch" wording
    # was already fairly strong (v1 results were clean) — kept as-is, only
    # tightened building/road wording for consistency with the other two
    # tiles' rework.
    # Reworked again pre-emptively before this tile's first run: both tile 1
    # (road-swallows-runway) and tile 2 (grass-swallows-railway/platform/
    # road) hit the same failure — a generic road or grass prompt was too
    # permissive and won the argmax across paved/mixed-texture areas that
    # should've belonged to a more specific class. This tile has the same
    # generic "paved road, street, asphalt road" and "tree, low vegetation,
    # grass" phrasing, so applying the same narrowing fix before the first
    # run rather than waiting to discover it again on a 3rd tile.
    # v13 result: even with the pre-emptive road/grass narrowing, `road`
    # sat at 0.00% in the multi-synonym baseline (19.7-19.8% in single-word/
    # ambiguous) — losing entirely to background, not to another class
    # (94.3% of single-word's road pixels read as background in the
    # multi-synonym run). Separately, user reviewed the A (window-size)
    # sweep directly and found a clean, reproducible pattern on the lake:
    # crop=768/stride=576 gives full, accurate water coverage; crop=1024/
    # stride=768 (then-baseline) already loses part of the lake's east edge
    # to `trees`; crop=1500/stride=1200 loses roughly half the same lake to
    # `trees`. Monotonic relationship, same prompt across all three, so this
    # is purely a window-size effect — a large uniform low-contrast surface
    # (dark reflective water under tree shadow) loses more resolution/
    # confidence per pixel as the crop grows, and the pixel falls to
    # whatever nearby class is more textured. Fix: gave this tile its own
    # `baseline` override at 768/576 (only water was checked against this
    # setting so far — road/pitch/court/building not yet re-verified at
    # this crop/stride, flagged as still open).
    # User then asked whether going even smaller has any downside before
    # testing further — answered (compute cost grows faster than linearly
    # as crop shrinks, less context per crop can fragment large uniform
    # objects, more seams to blend even with Gaussian weighting,
    # diminishing accuracy return past a point per published tile-inference
    # benchmarks) and confirmed this is a real balance to find empirically,
    # not assume. Added a dedicated smaller sweep point (512/384) via this
    # tile's own `window_sweep` override so Image A directly shows whether
    # going below the already-fixed 768/576 baseline helps or hurts other
    # classes (pitch/court/road/building), rather than re-testing the same
    # 3 points already on file. window_sweep intentionally does NOT change
    # any other tile's WINDOW_SIZE_SWEEP.
    "dop20_32_476_5524_1_he": {
        "baseline": dict(prob_thd=0.1, confidence_threshold=0.1, slide_crop=768, slide_stride=576),
        "window_sweep": [(512, 384), (768, 576), (1024, 768)],
        "multi": ["water body, river, lake",
                   "building, rooftop, residential roof",
                   "public paved road with painted lane markings and curbs, street with passing cars",
                   "green sports field, football pitch, grass pitch",
                   "clay sports court, dirt sports ground, tennis court",
                   "dense tree canopy, tree, forest, green lawn, bright green grass blades",
                   "car, vehicle"],
        "display": ["water", "building", "road", "pitch", "court", "trees", "vehicles"],
        "single": ["water", "building", "road", "pitch", "court", "tree", "car"],
        "ambiguous": ["water", "structure", "paved surface", "grass", "grass", "grass", "vehicle"],
        "colors": np.array([
            [  0, 100, 255],
            [  0,   0, 150],
            [ 80,  80,  80],
            [  0, 200,   0],
            [180, 100,  40],
            [  0, 255, 255],
            [255, 255,   0],
        ], dtype=np.uint8),
    },
    # New tile, added for parallel testing alongside tile 3. Confirmed via
    # a Kaggle dataset directory listing (not previously configured
    # anywhere) that this tile lives in `dummyirl/darmstadt-dop20` — the
    # SAME dataset as tile 3 (476_5524), adjacent in tile-grid coordinates
    # (473 vs 476, same 5524 row) — despite the user describing the scene
    # as "kindof hbf railstation"-like, which describes what it visually
    # resembles, not which dataset/city it's actually in.
    # User's own description of the scene (ground truth for class list,
    # not inferred from filename/thumbnail): grass/low vegetation, trees,
    # a rail station area with tracks and platforms extending from a
    # building structure, trains present, buildings, parking areas,
    # vehicles, and — explicitly NOT overexposed (unlike tile 2, so tile
    # 2's lighting-specific road/platform wording should NOT be reused
    # here by default). Also 3 bridges: one over a track, one over a road
    # (user noted this one is red-colored), and one more over a track
    # described as "like a walking bridge" (likely a pedestrian
    # footbridge, distinct from the road-carrying one).
    # Class-list decisions, each confirmed explicitly by the user rather
    # than assumed (this tile has an unusually high number of visually
    # similar paved/structural surfaces — road, platform, bridge deck x3,
    # parking area — plus the exact railway/platform/train combination
    # that already caused 2 full iteration cycles of failure+fix on tile
    # 2, so guessing here risked repeating that same class-confusion
    # pattern from scratch):
    # (1) platform tested SEPARATE from tracks this time, not merged —
    #     tile 2 only merged them because overexposure made them visually
    #     indistinguishable; this tile has normal lighting so that reason
    #     may not apply, worth testing fresh rather than assuming.
    # (2) parking area kept as its own class, distinct from road and
    #     vehicles, per user's explicit separation of these three concepts
    #     in their own description.
    # (3) bridge starts as ONE shared class for all 3 bridges (not
    #     pre-split by type) — split later only if the render shows real
    #     cross-type confusion, per the session's standing rule of testing
    #     before assuming rather than pre-guessing a fix.
    # All wording below either reuses tile 2's already-proven prompts
    # verbatim (train, trees, grass) or applies the tile-1/tile-3 road-
    # narrowing pattern (lane markings/curbs, NOT overexposure wording).
    # `parking area` and `bridge` are the only genuinely new prompts with
    # no prior data — kept deliberately short, consistent with the
    # session's finding that short wording has won more often than not on
    # rail/station scenes so far.
    "dop20_32_473_5524_1_he": {
        "multi": ["building",
                   "road with lane markings and curbs, street with passing cars",
                   "parking area, parking lot",
                   "car",
                   "station platform",
                   "railway track, rail line",
                   "train",
                   "bridge",
                   "tree, wooded canopy",
                   "grass, lawn, low vegetation"],
        "display": ["building", "road", "parking area", "vehicles", "platform",
                    "tracks", "train", "bridge", "trees", "grass"],
        "single": ["building", "road", "parking", "car", "platform",
                   "railway", "train", "bridge", "tree", "grass"],
        "ambiguous": ["structure", "paved surface", "paved surface", "vehicle",
                      "platform", "line", "vehicle", "structure", "grass", "grass"],
        "colors": np.array([
            [  0,   0, 255],   # building — blue
            [ 60,  60,  60],   # road — dark grey
            [140, 140,  40],   # parking area — olive
            [255, 255,   0],   # car — yellow
            [160, 160, 160],   # platform — light grey
            [128,   0, 128],   # tracks — purple
            [255, 140,   0],   # train — dark orange
            [220,  20,  60],   # bridge — crimson (distinct from road/platform greys)
            [  0, 200,   0],   # trees — green
            [  0, 255, 255],   # grass — cyan
        ], dtype=np.uint8),
    },
    # Added for direct comparability with a teammate's own run on this exact
    # tile (solar farm / rural Vogelsbergkreis-Lautertal scene). Uses their
    # published params verbatim (prob_thd=0.05, conf_thd=0.35, slide_stride=
    # 1200, slide_crop=1500) as this tile's own "baseline" override, plus
    # their exact class list/colors, so the baseline render is a like-for-
    # like comparison — no downsampling anywhere in this pipeline (confirmed:
    # no resize/downscale call exists in NB09), same as their base script.
    # Window-size (A) and threshold (B) sweeps still run around our own
    # 3-point ranges on top of this, same as every other tile, so we also
    # get the crop/stride and prob/conf sensitivity story for this scene.
    "DOP20_32_525_5604_1_he": {
        "baseline": dict(prob_thd=0.05, confidence_threshold=0.35, slide_crop=1500, slide_stride=1200),
        "multi": ["building, rooftop, residential roof",
                   "grassy field, low vegetation, meadow",
                   "paved road, street, farm track",
                   "car, vehicle, truck",
                   "tree, forest",
                   "solar panel array, photovoltaic panel, dark blue panel grid"],
        "display": ["building", "grass", "road", "vehicles", "trees", "solar panel"],
        "single": ["building", "grass", "road", "car", "tree", "solar panel"],
        "ambiguous": ["structure", "grass", "paved surface", "vehicle", "grass", "panel"],
        "colors": np.array([
            [  0,   0, 255],   # building — blue
            [124, 252,   0],   # grass — lawn green
            [128, 128, 128],   # road — grey
            [255, 255,   0],   # car/vehicle — yellow
            [ 34, 139,  34],   # tree — forest green
            [255, 165,   0],   # solar panel — orange
        ], dtype=np.uint8),
    },
}

if TARGET_TILE:
    if TARGET_TILE not in IMAGE_CONFIGS:
        print(f"ERROR: TARGET_TILE={TARGET_TILE!r} not in IMAGE_CONFIGS: {list(IMAGE_CONFIGS)}", flush=True)
        raise SystemExit(1)
    IMAGE_CONFIGS = {TARGET_TILE: IMAGE_CONFIGS[TARGET_TILE]}
print(f"Running tile(s): {list(IMAGE_CONFIGS)}", flush=True)

TILE_DATASET = {
    "dop20_32_468_5543_1_he": "frankfurt",
    "dop20_32_475_5550_1_he": "frankfurt",
    "dop20_32_476_5524_1_he": "darmstadt",
    "dop20_32_473_5524_1_he": "darmstadt",
    "DOP20_32_525_5604_1_he": "vogelsbergkreis-lautertal",
}

# ── resolve dataset input folders across all attached datasets ──
# Kaggle's mount path varies by how many/which datasets are attached — glob
# search rather than hardcode, same approach as NB08. Case-insensitive since
# tile 4's filename is uppercase-prefixed (DOP20_...) unlike tiles 1-3.
def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    if not hits:
        hits = sorted(p for p in Path("/kaggle/input").rglob("*.jpg")
                      if p.stem.lower() == stem.lower())
    return hits[0] if hits else None

resolved_paths = {}
for stem in IMAGE_CONFIGS:
    p = find_tile(stem)
    if p is None:
        print(f"NOT FOUND: {stem}.jpg (expected in dataset '{TILE_DATASET[stem]}-dop20')", flush=True)
    else:
        print(f"Resolved {stem} -> {p}", flush=True)
    resolved_paths[stem] = p

if all(v is None for v in resolved_paths.values()):
    print("\nERROR: no target tiles found anywhere under /kaggle/input. Listing what IS there:", flush=True)
    for p in sorted(Path("/kaggle/input").rglob("*.jpg"))[:50]:
        print(" ", p, flush=True)
    raise SystemExit(1)

# ── load model ──
from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

def make_processor(conf_thd):
    return Sam3Processor(model, confidence_threshold=conf_thd, device=DEVICE)

def cache_text(processor, words):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

def collect_class_scores(processor, state, h, w, te_cache, n_classes, device):
    logits = torch.zeros((n_classes, h, w), device=device)
    for cls_idx, te_cpu in enumerate(te_cache):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(device)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        scores = torch.zeros((h, w), device=device)
        if state.get("masks_logits") is not None and state["masks_logits"].shape[0] > 0:
            for i in range(state["masks_logits"].shape[0]):
                il = state["masks_logits"][i].squeeze()
                if il.shape != (h, w):
                    il = F.interpolate(il.view(1,1,*il.shape), size=(h,w),
                                       mode="bilinear", align_corners=False).squeeze()
                scores = torch.max(scores, il * state["object_score"][i])
        sem = state["semantic_mask_logits"].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(sem.view(1,1,*sem.shape), size=(h,w),
                                mode="bilinear", align_corners=False).squeeze()
        scores = torch.max(scores, sem) * state["presence_score"]
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

def make_gaussian_kernel(h, w, dev):
    sy, sx = h/4.0, w/4.0
    y = torch.arange(h, device=dev).float() - (h-1)/2.0
    x = torch.arange(w, device=dev).float() - (w-1)/2.0
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

def run_sliding_window(img_arr, words, processor, crop_size, stride):
    """One full sliding-window pass -> per-class logit map (before prob_thd)."""
    te_cache = cache_text(processor, words)
    n_cls = len(words)
    H_full, W_full = img_arr.shape[:2]

    h_grids = max(H_full - crop_size + stride - 1, 0) // stride + 1
    w_grids = max(W_full - crop_size + stride - 1, 0) // stride + 1
    total = h_grids * w_grids

    gauss_k = make_gaussian_kernel(crop_size, crop_size, DEVICE)
    acc     = torch.zeros(n_cls, H_full, W_full, device=DEVICE)
    wt_mat  = torch.zeros(H_full, W_full, device=DEVICE)

    for hi in range(h_grids):
        for wi in range(w_grids):
            y1 = hi*stride;  x1 = wi*stride
            y2 = min(y1+crop_size, H_full);  x2 = min(x1+crop_size, W_full)
            y1 = max(y2-crop_size, 0);       x1 = max(x2-crop_size, 0)

            crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])
            h_c, w_c = y2-y1, x2-x1

            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                state = processor.set_image(crop_pil)
                l = collect_class_scores(processor, state, h_c, w_c, te_cache, n_cls, DEVICE).float()

            g = gauss_k[:h_c, :w_c]
            acc[:, y1:y2, x1:x2] += l * g.unsqueeze(0)
            wt_mat[y1:y2, x1:x2] += g

            done = hi*w_grids + wi + 1
            print(f"    crop {done}/{total}", flush=True)

    return acc / wt_mat.unsqueeze(0)   # normalized per-class logit map, prob_thd NOT yet applied

def finalize(prob_map, prob_thd, bg_idx=BG_IDX):
    seg = prob_map.argmax(0)
    seg[prob_map.max(0)[0] < prob_thd] = bg_idx
    return seg.cpu().numpy()

# ── save raw prediction + metadata only. NO plotting/rendering here — that's
# a separate, GPU-free cell below so legend/layout tweaks never require a
# rerun of the (expensive) inference above. ──
manifest = []

def save_pred(stem, tag, seg, prob_thd, conf_thd, slide_stride, slide_crop):
    np.save(str(PRED_DIR / f"{stem}_{tag}.npy"), seg.astype(np.uint8))
    manifest.append(dict(stem=stem, tag=tag, prob_thd=prob_thd, conf_thd=conf_thd,
                          slide_stride=slide_stride, slide_crop=slide_crop))
    print(f"  Saved pred: {tag}", flush=True)

def run_image(stem, cfg, img_path):
    img_arr = np.array(Image.open(img_path).convert("RGB"))
    multi_words = cfg["multi"]
    img_size = (img_arr.shape[1], img_arr.shape[0])
    baseline = cfg.get("baseline", DEFAULT_BASELINE)
    window_sweep = cfg.get("window_sweep", WINDOW_SIZE_SWEEP)

    print(f"\n=== {stem} ===", flush=True)

    # ── baseline pass (multi-synonym prompts @ this tile's baseline params) — cached, reused everywhere ──
    print(f"  [baseline] multi-synonym prompts @ {baseline}", flush=True)
    proc_baseline = make_processor(baseline["confidence_threshold"])
    logits_baseline = run_sliding_window(img_arr, multi_words, proc_baseline,
                                          baseline["slide_crop"], baseline["slide_stride"])
    seg_baseline = finalize(logits_baseline, baseline["prob_thd"])
    save_pred(stem, "baseline", seg_baseline, baseline["prob_thd"], baseline["confidence_threshold"],
               baseline["slide_stride"], baseline["slide_crop"])

    # ═══ Image reference A: window-size (slide_crop/slide_stride) effect ═══
    for crop, stride in window_sweep:
        if (crop, stride) == (baseline["slide_crop"], baseline["slide_stride"]):
            seg = seg_baseline
        else:
            print(f"  [A-window] slide_crop={crop} slide_stride={stride} (rerun)", flush=True)
            lg = run_sliding_window(img_arr, multi_words, proc_baseline, crop, stride)
            seg = finalize(lg, baseline["prob_thd"])
        save_pred(stem, f"A_crop{crop}_stride{stride}", seg,
                   baseline["prob_thd"], baseline["confidence_threshold"], stride, crop)

    # ═══ Image reference B: prob_thd / confidence_threshold effect ═══
    for pt in PROB_THD_SWEEP:  # free — reuse logits_baseline
        seg = seg_baseline if pt == baseline["prob_thd"] else finalize(logits_baseline, pt)
        save_pred(stem, f"B_probthd{pt}", seg, pt, baseline["confidence_threshold"],
                   baseline["slide_stride"], baseline["slide_crop"])
    for ct in CONF_THD_SWEEP:  # real rerun — conf_thd changes SAM3's grounding output
        if ct == baseline["confidence_threshold"]:
            seg = seg_baseline
        else:
            print(f"  [B-conf] confidence_threshold={ct} (rerun)", flush=True)
            proc = make_processor(ct)
            lg = run_sliding_window(img_arr, multi_words, proc, baseline["slide_crop"], baseline["slide_stride"])
            seg = finalize(lg, baseline["prob_thd"])
        save_pred(stem, f"B_confthd{ct}", seg, baseline["prob_thd"], ct,
                   baseline["slide_stride"], baseline["slide_crop"])

    # ═══ Image reference C: prompt-wording effect (single-word / multi-synonym / ambiguous) ═══
    prompt_variants = {"multi-synonym": (multi_words, logits_baseline)}
    print("  [C-prompt] single-word prompts (rerun)", flush=True)
    prompt_variants["single-word"] = (cfg["single"], run_sliding_window(
        img_arr, cfg["single"], proc_baseline, baseline["slide_crop"], baseline["slide_stride"]))
    print("  [C-prompt] ambiguous prompts (rerun)", flush=True)
    prompt_variants["ambiguous"] = (cfg["ambiguous"], run_sliding_window(
        img_arr, cfg["ambiguous"], proc_baseline, baseline["slide_crop"], baseline["slide_stride"]))
    for label, (words, lg) in prompt_variants.items():
        seg = seg_baseline if label == "multi-synonym" else finalize(lg, baseline["prob_thd"])
        save_pred(stem, f"C_prompt_{label}", seg, baseline["prob_thd"], baseline["confidence_threshold"],
                   baseline["slide_stride"], baseline["slide_crop"])
        print(f"    {label} prompts used: {words}", flush=True)

    # per-tile metadata the rendering cell needs: display labels + colors + img_size
    (PRED_DIR / f"{stem}_meta.json").write_text(json.dumps(dict(
        display=cfg["display"], colors=cfg["colors"].tolist(), img_size=img_size)))

for stem, cfg in IMAGE_CONFIGS.items():
    img_path = resolved_paths[stem]
    if img_path is None:
        print(f"Skipping {stem} — file not found", flush=True)
        continue
    run_image(stem, cfg, img_path)

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\nWrote manifest.json with {len(manifest)} entries", flush=True)
PYEOF

## 4 — Render output images (no GPU, no SAM3 — edit freely and rerun)

Reads `output/preds/*.npy` + `output/manifest.json` + `output/{stem}_meta.json`
written by section 3. Pure matplotlib — safe to tweak legend/layout/colors
and rerun this cell alone; never re-triggers inference. Matches the
teammate's reference layout: RGB | overlay, title has no "α=" clutter beyond
the plot title itself, legend uses short display labels only (never the long
multi-synonym grounding strings), class-color swatch grid, params footer.

In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUT_DIR  = Path("/kaggle/working/output")
PRED_DIR = OUT_DIR / "preds"
BG_IDX = 255

TILE_DATASET = {
    "dop20_32_468_5543_1_he": "frankfurt",
    "dop20_32_475_5550_1_he": "frankfurt",
    "dop20_32_476_5524_1_he": "darmstadt",
    "dop20_32_473_5524_1_he": "darmstadt",
    "DOP20_32_525_5604_1_he": "vogelsbergkreis-lautertal",
}

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    if not hits:
        hits = sorted(p for p in Path("/kaggle/input").rglob("*.jpg")
                      if p.stem.lower() == stem.lower())
    return hits[0] if hits else None

def to_rgb(seg, color_map, bg_idx=BG_IDX):
    out = np.zeros((*seg.shape, 3), dtype=np.uint8)
    safe = np.where(seg == bg_idx, 0, seg)
    out[:] = color_map[np.clip(safe, 0, len(color_map)-1)]
    out[seg == bg_idx] = [30, 30, 30]
    return out

def render_result(stem, img_arr, seg, color_map, display_labels, out_path,
                   img_size, prob_thd, conf_thd, slide_stride, slide_crop, alpha=0.6):
    """Matches the team's exact reference layout code verbatim (figsize,
    subplots_adjust margins, legend position, separator line, meta_text
    format, dpi=200, bbox_inches='tight') so output is pixel-comparable."""
    fig = plt.figure(figsize=(20, 12))
    ax = fig.subplots(1, 2)

    ax[0].imshow(img_arr)
    ax[0].axis('off')
    ax[0].set_title(f"{stem}.jpg", fontsize=12, fontweight='bold')

    ax[1].imshow(img_arr)
    ax[1].imshow(to_rgb(seg, color_map), alpha=alpha)
    ax[1].axis('off')
    ax[1].set_title(f'Segmentation Result (α={alpha})', fontsize=12, fontweight='bold')

    legend_elements = []
    for class_name, color in zip(display_labels, color_map):
        legend_elements.append(Patch(facecolor=color / 255.0, edgecolor='black', label=class_name))

    plt.subplots_adjust(left=0.01, right=0.99, top=0.95, bottom=0.15, wspace=0.01)

    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.075),
               frameon=False, fontsize=9, ncol=min(5, len(display_labels)), prop={'weight': 'bold'})

    fig.lines.append(plt.Line2D([0.02, 0.98], [0.055, 0.055], transform=fig.transFigure,
                                 color='lightgray', linewidth=1))

    meta_text = (f"img_size = {img_size[0]}x{img_size[1]}    "
                 f"prob_thd = {prob_thd}    "
                 f"conf_thd = {conf_thd}    "
                 f"slide_stride = {slide_stride}    "
                 f"slide_crop = {slide_crop}")
    fig.text(0.5, 0.025, meta_text, ha='center', fontsize=10, family='monospace')

    fig.savefig(str(out_path), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  Rendered: {out_path.name}", flush=True)

manifest = json.loads((OUT_DIR / "manifest.json").read_text())
img_cache = {}
meta_cache = {}

for entry in manifest:
    stem, tag = entry["stem"], entry["tag"]
    if stem not in img_cache:
        p = find_tile(stem)
        img_cache[stem] = np.array(Image.open(p).convert("RGB"))
        meta_cache[stem] = json.loads((PRED_DIR / f"{stem}_meta.json").read_text())
    img_arr = img_cache[stem]
    meta = meta_cache[stem]
    seg = np.load(str(PRED_DIR / f"{stem}_{tag}.npy"))
    color_map = np.array(meta["colors"], dtype=np.uint8)
    render_result(stem, img_arr, seg, color_map, meta["display"],
                  OUT_DIR / f"{stem}_{tag}.png", meta["img_size"],
                  entry["prob_thd"], entry["conf_thd"], entry["slide_stride"], entry["slide_crop"])

print(f"\nRendered {len(manifest)} images from cached predictions.", flush=True)

### Takeaways — sensitivity runs

_Fill in after each run: which knob moved the result most, and where prompt
wording caused visible class confusion._

- `dop20_32_468_5543_1_he` (Frankfurt airport):
- `dop20_32_475_5550_1_he` (Frankfurt Hbf):
- `dop20_32_476_5524_1_he` (Darmstadt sports/water):